In [0]:
from pyspark.sql import functions as F

episodios = spark.table("mvp_engdados.bronze.episodes_raw")
imdb = spark.table("mvp_engdados.bronze.imdb_raw")

In [0]:
episodios_corrigidos = episodios.withColumn(
    "episode_num_in_season",
    F.when(
        (F.col("series_id") == "lost") &
        (F.col("season").cast("int") == 6) &
        (F.col("episode_num_overall").cast("int") == 105) &
        (F.col("title") == "LA X - Part 2"),
        F.lit("2")
    ).otherwise(F.col("episode_num_in_season"))
)

In [0]:
episodes_silver = (
    episodios_corrigidos
    .withColumn(
        "episode_num_in_season_raw",
        F.col("episode_num_in_season")
    )
    .withColumn(
        "_episode_num_in_season_clean",
        F.regexp_replace(
            F.trim(F.col("episode_num_in_season")),
            r"\[[0-9]+\]",
            ""
        )
    )
    .withColumn(
        "season",
        F.expr("try_cast(trim(season) AS INT)")
    )
    .withColumn(
        "episode_num_in_season",
        F.expr("try_cast(_episode_num_in_season_clean AS INT)")
    )
    .withColumn(
        "episode_num_overall",
        F.expr("try_cast(trim(episode_num_overall) AS INT)")
    )
    .withColumn(
        "us_viewers",
        F.expr("try_cast(nullif(trim(us_viewers), '') AS DOUBLE)")
    )
    .withColumnRenamed("original_air_date", "original_air_date_raw")
    .withColumn(
        "original_air_date",
        F.to_date("original_air_date_raw", "yyyy-MM-dd")
    )
    .drop("_episode_num_in_season_clean")
)

In [0]:
display(
    episodes_silver
    .filter(
        F.col("episode_num_in_season").isNull() &
        F.col("episode_num_in_season_raw").isNotNull()
    )
    .select(
        "series_id",
        "season",
        "episode_num_in_season_raw",
        "episode_num_overall",
        "title"
    )
)

series_id,season,episode_num_in_season_raw,episode_num_overall,title


In [0]:
(
    episodes_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.silver.episodes")
)

(
    imdb_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.silver.imdb")
)

In [0]:
episodes_silver.printSchema()
imdb_silver.printSchema()

display(
    spark.table("mvp_engdados.silver.episodes")
    .groupBy("series_id")
    .count()
    .orderBy("series_id")
)

display(
    spark.table("mvp_engdados.silver.imdb")
    .groupBy("series_id")
    .count()
    .orderBy("series_id")
)

root
 |-- season: integer (nullable = true)
 |-- episode_num_in_season: integer (nullable = true)
 |-- episode_num_overall: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- directed_by: string (nullable = true)
 |-- written_by: string (nullable = true)
 |-- original_air_date_raw: string (nullable = true)
 |-- prod_code: string (nullable = true)
 |-- us_viewers: double (nullable = true)
 |-- series_id: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- featured_character: string (nullable = true)
 |-- episode_num_in_season_raw: string (nullable = true)
 |-- original_air_date: date (nullable = true)

root
 |-- season: integer (nullable = true)
 |-- episode_num: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- original_air_date_raw: string (nullable = true)
 |-- imdb_rating: double (nullable = true)
 |-- total_votes: long (nullable = true)
 |-- desc: string (nullable = true)
 |-- se

series_id,count
big_bang_theory,279
breaking_bad,62
brooklyn_99,153
friends,236
game_of_thrones,73
how_i_met_your_mother,208
lost,121
the_office,201
the_walking_dead,177


series_id,count
big_bang_theory,280
breaking_bad,62
brooklyn_99,153
friends,235
game_of_thrones,73
how_i_met_your_mother,208
lost,119
the_office,188
the_walking_dead,177
